# DriftSense full-session model

## tl;dr

- Trained on **634 usable sessions** from **19 participants**.
- Repeated participant-grouped development selected `activity_only` with `C=0.1`.
- On **193 later sessions**, ROC-AUC was **0.931** (participant-bootstrap 95% CI **0.891–0.962**), precision was **0.907**, recall was **0.773**, and F1 was **0.834** at the development-selected threshold.
- The final JSON artifact matches the Python pipeline to a maximum absolute probability error of **5.55e-17** across shared test vectors.


## Context & Methods

The model predicts the later binary post-session alignment answer from information available after the task session ends. It does not use the answer itself. Candidate feature families and regularization were selected with five repeats of participant-grouped five-fold validation on participant-relative days 1–7. The selected configuration and threshold were evaluated once on later sessions.

### Key Assumptions

- Session rows are the unit of analysis and participant IDs define validation groups.
- Prior-session calibration features use only earlier session outcomes and activity.
- The positive-decision cap is 35% in development data.
- Participant IDs are never predictive inputs.


## Data

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from ml.full_session_model import run_full_session_training

SESSIONS = Path('D:\\1.msc\\DriftSense\\driftsense_merged.csv')
ARTIFACTS = Path('D:\\1.msc\\DriftSense\\ml\\artifacts\\full_session_model')

summary = run_full_session_training(
    sessions_path=SESSIONS,
    output_directory=ARTIFACTS,
    development_days=7,
    repeats=5,
    folds=5,
    max_positive_rate=0.35,
)
quality = summary["data_quality"]
display(pd.DataFrame([{
    "sessions": quality["rows"],
    "participants": quality["participants"],
    "usable_labels": quality["usable_binary_labels"],
    "excluded_uncertain_or_missing": quality["excluded_uncertain_or_missing_labels"],
    "drift_prevalence": quality["drift_prevalence"],
    "duplicate_session_ids": quality["duplicate_session_ids"],
    "overlapping_sessions": quality["overlapping_sessions"],
}]))
display(pd.DataFrame([summary["split"]]))


,sessions,participants,usable_labels,excluded_uncertain_or_missing,drift_prevalence,duplicate_session_ids,overlapping_sessions
0,665,19,634,31,0.435331,0,0


,development_days,development_rows,development_participants,chronological_holdout_rows,chronological_holdout_participants,final_training_rows
0,7,441,19,193,18,634


## Results

In [2]:
tuning = pd.read_csv(ARTIFACTS / "full_session_tuning.csv")
best_tuning = (
    tuning.sort_values(["model", "roc_auc", "brier"], ascending=[True, False, True])
    .groupby("model", as_index=False)
    .first()
)
display(best_tuning[["model", "regularization_c", "roc_auc", "brier", "f1", "prompt_rate"]].sort_values("roc_auc", ascending=False))

comparison = pd.read_csv(ARTIFACTS / "full_session_model_comparison.csv")
holdout = comparison[comparison["evaluation"] == "chronological_known_participant_holdout"]
display(holdout[["model", "regularization_c", "n", "roc_auc", "brier", "accuracy", "precision", "recall", "f1", "prompt_rate"]].sort_values("roc_auc", ascending=False))

display(pd.read_csv(ARTIFACTS / "full_session_calibration.csv"))
display(pd.read_csv(ARTIFACTS / "full_session_coefficients.csv").head(15))
display(pd.DataFrame([summary["chronological_holdout"]]))


,model,regularization_c,roc_auc,brier,f1,prompt_rate
0,activity_only,0.10,0.925385,0.109850,0.794521,0.401361
1,context_activity,0.03,0.920675,0.118176,0.795640,0.405896
2,context_activity_time,0.03,0.920129,0.118906,0.793388,0.396825
5,participant_calibrated_context_activity,0.03,0.916281,0.118825,0.780220,0.399093
3,context_only,0.30,0.552140,0.247321,0.369637,0.260771
6,task_site_domain_only,0.30,0.547578,0.243079,0.326848,0.156463
7,task_type_only,1.00,0.546916,0.245817,0.346021,0.229025
4,intended_duration_only,3.00,0.511605,0.245997,0.030928,0.013605


,model,regularization_c,n,roc_auc,brier,accuracy,precision,recall,f1,prompt_rate
19,participant_calibrated_context_activity,0.03,193,0.936147,0.103207,0.849741,0.824176,0.852273,0.837989,0.471503
17,context_activity_time,0.03,193,0.935823,0.106876,0.860104,0.842697,0.852273,0.847458,0.461140
15,context_activity,0.03,193,0.932468,0.109851,0.854922,0.840909,0.840909,0.840909,0.455959
13,activity_only,0.10,193,0.930519,0.108855,0.834197,0.804348,0.840909,0.822222,0.476684
11,context_only,0.30,193,0.631061,0.238141,0.616580,0.620690,0.409091,0.493151,0.300518
9,task_type_only,1.00,193,0.619643,0.239971,0.606218,0.630435,0.329545,0.432836,0.238342
7,task_site_domain_only,0.30,193,0.576190,0.244174,0.580311,0.652174,0.170455,0.270270,0.119171
5,intended_duration_only,3.00,193,0.550920,0.246359,0.559585,0.800000,0.045455,0.086022,0.025907
3,fixed_timer_prompt_all,NaN,193,0.500000,0.544041,0.455959,0.455959,1.000000,0.626335,1.000000
2,majority_class,NaN,193,0.500000,0.248940,0.544041,0.000000,0.000000,0.000000,0.000000


,sessions,mean_predicted_probability,observed_drift_rate,probability_min,probability_max
0,39,0.031460,0.025641,0.001261,0.082260
1,38,0.164296,0.184211,0.088436,0.292212
2,39,0.463477,0.282051,0.299458,0.600786
3,38,0.774634,0.789474,0.605779,0.880480
4,39,0.946742,1.000000,0.884298,0.998197


,transformed_feature,coefficient,absolute_coefficient
0,active_share,-1.238123,1.238123
1,away_share,0.981849,0.981849
2,idle_share,0.621713,0.621713
3,keyboard_rate_per_min,-0.368549,0.368549
4,click_rate_per_min,0.252638,0.252638
5,log1p_away_seconds,0.232019,0.232019
6,tab_switch_rate_per_min,0.204098,0.204098
7,scroll_rate_per_min,-0.196227,0.196227
8,log1p_keyboard_activity_count,-0.133636,0.133636
9,log1p_idle_seconds,0.128137,0.128137


,n,prevalence,accuracy,precision,recall,f1,roc_auc,brier,prompt_rate,false_prompt_rate_all_sessions,false_prompt_share_of_prompts,threshold,tn,fp,fn,tp
0,193,0.455959,0.860104,0.906667,0.772727,0.834356,0.930519,0.108855,0.388601,0.036269,0.093333,0.612914,98,7,20,68


## Takeaways

1. Aggregate activity provides substantially more grouped-development discrimination than task context alone in this dataset.
2. Active time is represented by both active share and log-transformed active seconds; coefficient magnitude is associative, not causal importance.
3. The frozen threshold must be reported together with recall, positive-decision rate, and false-positive burden.
4. The artifact is appropriate for session-end research use. A separate cutoff feature export and validation run are required for a 3-, 5-, or 10-minute intervention model.
5. Participant-resampled intervals and calibration should accompany paper claims; a single accuracy value is insufficient.
